In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Part 1: Read Data
# Task 1: Read the dataset
print(os.listdir(path))
csv_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Inspect the first few rows
print("First 5 rows of the dataset:")
print(df.head())

In [ ]:
# Task 3: Display dataset information
print("\nDataset Information:")
print(df.info())

In [ ]:
# Task 4: Show statistical description
print("\nStatistical Description:")
print(df.describe())


In [ ]:
# Task 5: Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 6))
plt.hist(df['Delivery_Time'], bins=30, color='skyblue', edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Drop the 'Order_ID' column from the data
df_clean = df.copy()
df_clean.drop(columns=['Order_ID'])



In [ ]:
# Task 2: Handle missing values appropriately
def check_missing_values(df_clean):
  missing_values = df_clean.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


# Handle missing values (drop rows with missing target values)
df_cleaned = df_clean.dropna(subset=['Delivery_Time'])
print(df_cleaned.Delivery_Time.isna().sum())

df_clean = df_clean.dropna(subset=['Delivery_Time']).copy()




In [ ]:
# Task 3: Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

df_clean = df_clean.drop_duplicates().copy()

In [ ]:
# Encode ALL object columns
from sklearn.preprocessing import LabelEncoder

obj_cols = df_clean.select_dtypes(include=['object']).columns

for col in obj_cols:
    df_clean[col] = df_clean[col].fillna('Unknown').astype(str)
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])

    print(df_clean.select_dtypes(include=['object']).columns)  # لازم تطلع Index([]) يعني ما فيه نصوص



In [ ]:
# Task 5: Write your code here:
# I did it when Prepare Data for Modeling (in the bottom)

In [ ]:
# Task 6: Write your code here: NO Need

In [ ]:
# Task 1

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# 1) Split into X and y
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']


In [ ]:
# Task 2,3,4,5: KFold + RandomForest + MAE (avg across folds)

# 2) KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train_fold = X.iloc[train_idx]
    X_val_fold   = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold   = y.iloc[val_idx]

    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)

    preds = model.predict(X_val_fold)
    mae_scores.append(mean_absolute_error(y_val_fold, preds))

print("Average MAE across all folds:", np.mean(mae_scores))



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Train final model on full data (for plots)
final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
final_model.fit(X, y)

# 1) Feature importance
importances = final_model.feature_importances_
feat_names = X.columns

sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(feat_names[sorted_idx], importances[sorted_idx])
plt.title("Feature Importance (RandomForest)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



In [ ]:
# Task 2: Predicted delivery time histogram
y_pred_all = final_model.predict(X)

plt.figure(figsize=(10, 5))
plt.hist(y_pred_all, bins=30, edgecolor='black')
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
!pip install catboost


In [ ]:
# Task Bonus: Ensemble (RandomForest + CatBost) with KFold + MAE on averaged preds

import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train_fold = X.iloc[train_idx]
    X_val_fold   = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold   = y.iloc[val_idx]

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    cb = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        loss_function="MAE",
        random_seed=42,
        verbose=False
    )

    rf.fit(X_train_fold, y_train_fold)
    cb.fit(X_train_fold, y_train_fold)

    pred_rf = rf.predict(X_val_fold)
    pred_cb = cb.predict(X_val_fold)

    pred_avg = (pred_rf + pred_cb) / 2.0
    mae_scores.append(mean_absolute_error(y_val_fold, pred_avg))

print("Average MAE across all folds (Ensemble):", np.mean(mae_scores))
